# Inverse Procedure on Linear Elasticity Problem

In [ ]:
# !pip install git+https://github.com/Violandree/PODCNF.git # Da modificare
# It is not necessary to install the library if you download locally the repository

In [19]:
import os
import gdown
import numpy as np

import torch
import tqdm

from podcnf.NFmodel import NormalizingFlow

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

## Load Data

In [2]:
# Data
os.makedirs('../data', exist_ok=True)
target_folder = os.path.join("..", "data") 
filename = os.path.join(target_folder, "data_linear_density_2pi.pt")
gdown.download(id = "1NYiXCqCNLhB87o3OlAav6YfiAO9qoBnM", quiet=True, output = filename)
loaded_data = torch.load(filename)

# Test Data
filenameTest = os.path.join(target_folder, "data_test.pt")
gdown.download(id = "1GtBiXTjdzCFM16KTTMiAD7aiqNcu-ggc", quiet=True, output = filenameTest)
loaded_data_test = torch.load(filenameTest)

In [3]:
mass = loaded_data['mass']
delta = loaded_data['delta']
g = loaded_data['g_data']
u = loaded_data['u_data']

In [8]:
loaded_data.keys()

dict_keys(['mass', 'delta', 'g_data', 'u_data'])

In [4]:
mass_TEST = loaded_data_test['mass']
delta_TEST = loaded_data_test['delta']
theta_TEST = loaded_data_test['theta']
g_TEST = loaded_data_test['g_data']
u_TEST = loaded_data_test['u_data']

In [5]:
print(f"mass shape: {mass.shape}")
print(f"delta shape: {delta.shape}")
print(f"g shape: {g.shape}")
print(f"u shape: {u.shape}")

mass shape: torch.Size([6400, 1])
delta shape: torch.Size([6400, 1])
g shape: torch.Size([6400, 1800])
u shape: torch.Size([6400, 1922])


In [6]:
mu = torch.cat((mass, delta), 1)
mu_TEST = torch.cat((mass_TEST, delta_TEST), 1)
print(f"mu: {mu.shape}")

mu: torch.Size([6400, 2])


In [ ]:
import logging
logging.getLogger("FFC").setLevel(logging.WARNING)

n_samples = mass.shape[0]
index = np.random.randint(0, n_samples - 1)
plot_data(g_TEST[index], u_TEST[index], Vh, V_lam, mu_TEST[index], theta_TEST[index])

# Inverse Problem

Haario et al. (Adaptive Metropolis)

In questo notebook si prova ad investigare sulla natura dei parametri noti e sulla quale si è condizionato per trovare $p_{NF}(u_x,u_y|\mu)$.
Dato infatti il metodo forward si vuole ora determinare il metodo inverse, quello che dal modello trovato prova a ricostruire i valori di $m$ e $\delta$. Dal momento che in linea teorica i NF per come definito in questo caso di studio sono in grado di trovare la distribuzione vera $p(u_x,u_y|\mu)$ (vedi teoria NF), è naturale pensare che l'architettura migliore sia quella di un Metropolis-Hastings. In particolare, la soluzione proposta sarà quella di un Adaptive-MH.

In questo caso immaginiamo di aver allenato una volta per tutte il modello NF su tutta la mesh e succesivamente per fare il MH consideriamo di avere in nostro possesso solo i sensori di superficie. Quindi dato il modello allenato selezioniamo i sensori che ci interssano e fittiamo il MH.

## Load Model

In [ ]:
# {'learning_rate': 0.001, 'num_flows': 16, 'hidden_size': 256, 'hidden_depth': 2, 'weight_decacy': 1e-05}
os.makedirs('../results/elastics', exist_ok=True)
target_folder = os.path.join("..", "results/elastic")
MODEL_NAME = os.path.join(target_folder, 'MODEL_64_NEW.pth')
gdown.download(id = "1Mv9opjkEMDQaLBQx07Fqvh_ItyefLsCl", quiet=True, output = MODEL_NAME)
loaded_model = torch.load(MODEL_NAME, map_location=device)

In [ ]:
# Da creare il file su onedrive, i link è di Stokes
# os.makedirs('../data', exist_ok=True)
# target_folder = os.path.join("..", "data") 
# reduced_input_file = os.path.join(target_folder, "elastic_data_reduced_6400.pt")
# gdown.download(id="19304ojlsmuL7hntN-m8KeR_CrAZD7wHb", quiet=True, output=reduced_input_file)
# reduced_dataset = torch.load(reduced_input_file, weights_only=True)
# mu = reduced_dataset['mu'].numpy()
# c = reduced_dataset['c'].numpy()

In [13]:
# mu and c for the initialization of the model
reduced_input_file = "../data/elastic_data_reduced_6400.pt" 

reduced_dataset = torch.load(reduced_input_file, weights_only=True)
mu = reduced_dataset['mu'].numpy()
c = reduced_dataset['c'].numpy()

In [15]:
dim_x = mu.shape[1]
dim_y = c.shape[1]

In [16]:
# Linear model
num_flows = 16
hidden_size = 256
hidden_depth = 2

NF_linear = NormalizingFlow(dim_x, dim_y, num_flows, hidden_size, hidden_depth, device).to(device)
NF_linear.load_state_dict(loaded_model)

<All keys matched successfully>

In [ ]:
# Loaded Flow - Linear
flow = NF_linear

## Analysis

In [17]:
# Coordinates of all the dof (1922*2)
# dof_coords = Vh.tabulate_dof_coordinates()
# n_dofs = Vh.dim()
num_sensors = 31

In [20]:
sur = [0,1] # top-left already added
for j in range(num_sensors-1):
    xy = np.array([np.pow(j+2,2)+j, np.pow(j+2,2)+j+1])
    sur.extend(xy)
surface_idx = np.array(sur)
print(f"Number of sensors on the surface: {len(surface_idx)}")

Number of sensors on the surface: 62


In [21]:
u_surface_sensor = u[:, surface_idx] # selezione direttamente i sensori sulla superficie
print(u_surface_sensor.shape)

torch.Size([6400, 62])


Here the proposed method

In [22]:
def adaptive_metropolis_hastings(
    flow_model, u_obs, theta_0, N, bounds,
    mu_scaler, c_scaler, device,
    V, surface_idx,
    temperature=1.0, C_0=None, n_0=100, epsilon=1e-6, s_d=None,
    n_generations=100, h=0.1
):

    flow_model.eval()

    # Outside the loop I'll compute the "projection" matrix into the subspace (62x20)
    # In this way u_sensor is directly given by: u_sensor = c * V_sensor'
    V_sensors = V[surface_idx, :]
    V_sensors_torch = torch.tensor(V_sensors, dtype=torch.float32).to(device)

    u_obs_tensor = u_obs.to(device)
    mean_c = torch.tensor(c_scaler.mean_, dtype=torch.float32).to(device)
    scale_c = torch.tensor(c_scaler.scale_, dtype=torch.float32).to(device)

    # Initialization
    theta_n = np.array(theta_0, dtype=np.float32)
    d = len(theta_n)
    theta_n_scaled = mu_scaler.transform(theta_n.reshape(1, -1))
    theta_tensor = torch.tensor(theta_n_scaled, dtype=torch.float32).to(device)

    def compute_log_likelihood(theta_t):
        with torch.no_grad():
            theta_rep = theta_t.repeat(n_generations, 1)
            c_samples_norm = flow_model.sample(theta_rep) # sample ritorna (z, ldj), prendo z (c_samples)
            c_samples = c_samples_norm * scale_c + mean_c

            # Projection on the sensors
            # [N_gen, 20] @ [20, 62] -> [N_gen, 62]
            u_sensor_sample = c_samples @ V_sensors_torch.T

            # Da rivedere dal punto di vista teorico. In linea di prinicipio dal momento che il NF
            # non da un'unica soluzione per un dato valore di theta, considera n_generations (= 100)
            # sample, ognuno di questi fornisce una soluzione diversa. Per ogni soluzione fornita
            # dal NF nel calcolo l'errore in norma L2 con u_obs_tensor. Ottenute le distanze utilizza
            # un Kernel Gaussiano per vedere la bontà del theta selezionato
            # L2-distance
            diff = u_sensor_sample - u_obs_tensor.reshape(1, -1)
            dj2 = diff.pow(2).sum(dim=1) # [N_gen]

            # LogSumExp per stabilità (KDE Likelihood)
            # log( sum(exp(-d^2 / 2h^2)) ) - log(N)
            log_pi = torch.logsumexp(-dj2 / (2 * h**2), dim=0) - np.log(n_generations)

            return log_pi.item()

    # Initial value for the Log-likelihood
    log_pi_n = compute_log_likelihood(theta_tensor)

    chain = []
    accepted_count = 0

    # COvariance
    if C_0 is None:
        C_n = np.eye(d) * 1e-5
    else:
        C_n = np.array(C_0)

    theta_bar_n = theta_n.copy()

    # Scaling factor
    if s_d is None:
        scaling_val = (2.38**2) / d
    else:
        scaling_val = s_d

    # MCMC LOOP
    for n in tqdm(range(1, N + 1), desc="Adaptive MH"):

        # Proposal Covariance once n>n_0
        if n <= n_0:
            proposal_cov = C_n
        else:
            proposal_cov = scaling_val * C_n + epsilon * np.eye(d)

        perturbation = np.random.multivariate_normal(np.zeros(d), proposal_cov)
        Y = theta_n + perturbation

        # Check Prior (Bounds)
        if (Y[0] < bounds['m_min'] or Y[0] > bounds['m_max'] or
            Y[1] < bounds['d_min'] or Y[1] > bounds['d_max']):
            chain.append(theta_n)
            theta_next = theta_n
        else:
            # Compute the likelihood for the candidate
            Y_scaled = mu_scaler.transform(Y.reshape(1, -1))
            Y_tensor = torch.tensor(Y_scaled, dtype=torch.float32).to(device)

            log_pi_Y = compute_log_likelihood(Y_tensor)

            # Acceptance ratio
            log_alpha = (log_pi_Y - log_pi_n) / temperature

            if np.log(np.random.rand()) < log_alpha:
                theta_n = Y
                log_pi_n = log_pi_Y
                accepted_count += 1

            chain.append(theta_n)
            theta_next = theta_n

        # Updating the covariance
        if n > n_0:
            theta_bar_prev = theta_bar_n.copy()
            theta_bar_n = (n * theta_bar_prev + theta_next) / (n + 1)

            dt = (theta_next - theta_bar_prev).reshape(-1, 1)
            term_update = np.dot(dt, dt.T)
            C_n = ((n - 1) / n) * C_n + (scaling_val / n) * (term_update * (n / (n + 1)) + epsilon * np.eye(d))

    chain = np.array(chain)
    acc_rate = accepted_count / N
    print(f"\nAcceptance Rate: {acc_rate:.2%}")
    return chain, C_n

Due fasi, una di "exploration" e una seconda fase di raffinamento.

In [23]:
# Choose a number between 6081 and 6400
# test_idx = np.random.randint(n_val, n_samples)
# print(test_idx)

test_idx = 6304

In [24]:
from time import perf_counter

bounds = {
    'm_min': 1.0, 'm_max': 2.0,
    'd_min': 0.05, 'd_max': 0.25
}

mu_true_phys = mu[test_idx].numpy()
u_obs = u_surface_sensor[test_idx]

initial_guess = [np.random.rand() + 1, np.random.rand()*0.1 + 0.15]

print(f"Index: {test_idx} - True Mass={mu_true_phys[0]:.4f}, True Delta={mu_true_phys[1]:.4f}")
print(f"Initial guess: {initial_guess}")

print("\n--- EXPLORATION ---")
initial_cov_expl = [[0.001, 0], [0, 0.001]]

t0 = perf_counter()

chain_exploration, cov_learned = adaptive_metropolis_hastings(
    flow_model=flow,
    u_obs=u_obs,
    theta_0=initial_guess,
    N=10000,
    bounds=bounds,
    mu_scaler=mu_scaler,
    c_scaler=c_scaler,
    device=device,
    V=V,
    surface_idx=surface_idx,
    temperature=5.0,
    C_0=initial_cov_expl,
    n_0=500,
    s_d=2.4,
    n_generations=100,
    h=0.1
)

t_exp = perf_counter() - t0

best_guess = chain_exploration[-1]
print(f"Best Guess after exploration: {best_guess}")

print("\n--- REFINEMENT ---")
cov_for_refinement = cov_learned + np.eye(2) * 1e-6

t1 = perf_counter()

chain_refined, _ = adaptive_metropolis_hastings(
    flow_model=flow,
    u_obs=u_obs,
    theta_0=best_guess,
    N=30000,
    bounds=bounds,
    mu_scaler=mu_scaler,
    c_scaler=c_scaler,
    device=device,
    V=V,
    surface_idx=surface_idx,
    temperature=1.0,
    C_0=cov_for_refinement,
    n_0=0,
    s_d=0.35,
    n_generations=200,
    h=0.03
)

t_ref = perf_counter() - t1

AttributeError: 'numpy.ndarray' object has no attribute 'numpy'

In [ ]:
print(f">>>PODCNF\nExploration time:\t{t_exp}\nRefinement time:\t{t_ref}")

>PODCNF (6247)

Exploration time:	54.16307142799997 - Refinement time:	317.2200043969988

>PODCNF (6347)

Exploration time: 49.78578957298472 - Refinement time:	358.7658075280470

>PODCNF (6304)

Exploration time:	49.861697442000036 - Refinement time:	361.094460828

>PODCNF (6265)

Exploration time:	49.79213134499969 - Refinement time:	335.21530144999997

>PODCNF (6294)

Exploration time:	42.25949279700001 - Refinement time:	314.42771635699995

> PODCNF (6386)

Exploration time:	39.868961181000486 - Refinement time:	297.06175480299953

In [ ]:
full_chain = np.vstack([chain_exploration, chain_refined])
split_point = len(chain_exploration)
clean_samples = chain_refined[500:]

fig1, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(full_chain[:, 0], color='#1f77b4', linewidth=0.8, alpha=0.9, label='Chain')
axes[0].axvline(x=split_point, color='black', linestyle='--', linewidth=1.5, label='Refinement')
axes[0].axhline(y=mu_true_phys[0], color='red', linestyle='--', linewidth=2, label='True target')
axes[0].set_title(f"Trace Plot: Mass ({mu_true_phys[0]:.4f})", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Mass")
axes[0].set_xlabel("Iteration")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(full_chain[:, 1], color='#ff7f0e', linewidth=0.8, alpha=0.9, label='Chain')
axes[1].axvline(x=split_point, color='black', linestyle='--', linewidth=1.5)
axes[1].axhline(y=mu_true_phys[1], color='red', linestyle='--', linewidth=2, label='True target')
axes[1].set_title(f"Trace Plot: Delta ({mu_true_phys[1]:.4f})", fontsize=14, fontweight='bold')
axes[1].set_ylabel("Delta")
axes[1].set_xlabel("Iteration")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fig2, ax_post = plt.subplots(figsize=(7, 5))

# KDE Plot
sns.kdeplot(
    x=clean_samples[:, 0],
    y=clean_samples[:, 1],
    cmap="Blues",
    fill=True,
    thresh=0.05,
    levels=10,
    ax=ax_post
)

ax_post.scatter(mu_true_phys[0], mu_true_phys[1], s=300, c='red', marker='X', label='True target', zorder=10)
ax_post.scatter(clean_samples[::200, 0], clean_samples[::200, 1], s=5, c='black', alpha=0.1, label='Samples')
ax_post.set_title("Posterior Distribution", fontsize=14, fontweight='bold')
ax_post.set_xlabel("Mass")
ax_post.set_ylabel("Delta")
ax_post.legend()
ax_post.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
cleaner = 2000
mass_chain = chain_refined[cleaner:, 0]
delta_chain = chain_refined[cleaner:, 1]

print(f"True mass: {mu_true_phys[0]:.4f}")
print(f"Estimated Mass: {np.mean(mass_chain):.4f} +/- {np.std(mass_chain):.4f}")
print(f"Mass relative error: {abs(np.mean(mass_chain) - mu_true_phys[0])/mu_true_phys[0]:.4%}")

print("-" * 20)
print(f"True Delta: {mu_true_phys[1]:.4f}")
print(f"Estaimted Delta: {np.mean(delta_chain):.4f} +/- {np.std(delta_chain):.4f}")
print(f"Delta relative error: {abs(np.mean(delta_chain) - mu_true_phys[1])/mu_true_phys[1]:.4%}")

test_index = 6347

True mass: 1.1632 - Estimated Mass: 1.2819 +/- 0.1348 - Mass relative error: 10.2040%

True Delta: 0.2330 - Estaimted Delta: 0.2090 +/- 0.0218 - Delta relative error: 10.3067

test_index = 6247

True mass: 1.4902 - Estimated Mass: 1.5411 +/- 0.0760 - Mass relative error: 3.4159%

True Delta: 0.2342 - Estaimted Delta: 0.2356 +/- 0.0113 - Delta relative error: 0.5771%

test_index = 6304

True mass: 1.1811 - Estimated Mass: 1.2380 +/- 0.1241 - Mass relative error: 4.8181%

True Delta: 0.2132 - Estaimted Delta: 0.2092 +/- 0.0214 - Delta relative error: 1.8829%

test_idx = 6265

True mass: 1.6223 - Estimated Mass: 1.6303 +/- 0.1356 - Mass relative error: 0.4959%

True Delta: 0.1962 - Estaimted Delta: 0.1842 +/- 0.0164 - Delta relative error: 6.1438%

test_idx = 6294

True mass: 1.4037 - Estimated Mass: 1.4340 +/- 0.1198 - Mass relative error: 2.1614%

True Delta: 0.2213 - Estaimted Delta: 0.2155 +/- 0.0172 - Delta relative error: 2.6210%

test_idx = 6386

True mass: 1.8839 - Estimated Mass: 1.8812 +/- 0.0841 - Mass relative error: 0.1447%

True Delta: 0.1505 - Estaimted Delta: 0.1516 +/- 0.0133 - Delta relative error: 0.6923%

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from scipy.special import logsumexp
from time import perf_counter

import logging
from dolfin import set_log_level, LogLevel

set_log_level(LogLevel.WARNING)

logging.getLogger('FFC').setLevel(logging.WARNING)
logging.getLogger('UFL').setLevel(logging.WARNING)

def adaptive_metropolis_hastings_fom(
    u_obs, theta_0, N, bounds, surface_idx,
    temperature=1.0, C_0=None, n_0=100, epsilon=1e-6, s_d=None,
    n_generations=10, h=0.1
):

    u_obs_np = np.array(u_obs)

    # Initialization
    theta_n = np.array(theta_0, dtype=np.float64)
    d = len(theta_n)

    def compute_log_likelihood(theta_t):
        m_in, d_in = theta_t[0], theta_t[1]
        u_sensor_samples = np.zeros((n_generations, len(surface_idx)))

        # Marginalizing over the stochasticity of the PDE using the FOM
        for g_idx in range(n_generations):
            seed_in = np.random.randint(0, 2**32 - 1)

            # Generates data and select the sensors
            _, _, _, u_data_full, _ = FOMsampler(seed_in, m_in, d_in, option=1)
            u_sensor_samples[g_idx, :] = u_data_full[surface_idx]

        # L2 distance squared on sensors
        diff = u_sensor_samples - u_obs_np.reshape(1, -1)
        dj2 = np.sum(diff**2, axis=1) # [N_gen]

        # KDE Likelihood using LogSumExp for stability
        log_pi = logsumexp(-dj2 / (2 * h**2)) - np.log(n_generations)

        return log_pi

    # Initial value for the Log-likelihood
    print(f"Computing initial likelihood (requires {n_generations} FOM solves)...")
    log_pi_n = compute_log_likelihood(theta_n)
    print("Initial likelihood computed. Starting MCMC loop...")

    chain = []
    accepted_count = 0

    # Covariance initialization
    if C_0 is None:
        C_n = np.eye(d) * 1e-5
    else:
        C_n = np.array(C_0)

    theta_bar_n = theta_n.copy()

    # Scaling factor for the proposal distribution
    if s_d is None:
        scaling_val = (2.38**2) / d
    else:
        scaling_val = s_d

    # MCMC LOOP
    for n in tqdm(range(1, N + 1), desc="Adaptive MH (FOM)"):

        # Proposal Covariance once n > n_0
        if n <= n_0:
            proposal_cov = C_n
        else:
            proposal_cov = scaling_val * C_n + epsilon * np.eye(d)

        perturbation = np.random.multivariate_normal(np.zeros(d), proposal_cov)
        Y = theta_n + perturbation

        # Check Prior (Bounds)
        if (Y[0] < bounds['m_min'] or Y[0] > bounds['m_max'] or
            Y[1] < bounds['d_min'] or Y[1] > bounds['d_max']):
            chain.append(theta_n)
            theta_next = theta_n
        else:
            # Compute the likelihood for the candidate directly with the physical parameters
            log_pi_Y = compute_log_likelihood(Y)

            # Acceptance ratio
            log_alpha = (log_pi_Y - log_pi_n) / temperature

            if np.log(np.random.rand()) < log_alpha:
                theta_n = Y
                log_pi_n = log_pi_Y
                accepted_count += 1

            chain.append(theta_n)
            theta_next = theta_n

        # Updating the covariance
        if n > n_0:
            theta_bar_prev = theta_bar_n.copy()
            theta_bar_n = (n * theta_bar_prev + theta_next) / (n + 1)

            dt = (theta_next - theta_bar_prev).reshape(-1, 1)
            term_update = np.dot(dt, dt.T)

            # Update step
            C_n = ((n - 1) / n) * C_n + (1 / n) * (term_update * (n / (n + 1)) + epsilon * np.eye(d))

    chain = np.array(chain)
    acc_rate = accepted_count / N
    print(f"\nAcceptance Rate: {acc_rate:.2%}")
    return chain, C_n

In [ ]:
print(test_idx)

In [ ]:
bounds = {
    'm_min': 1.0, 'm_max': 2.0,
    'd_min': 0.05, 'd_max': 0.25
}

mu_true_phys = mu[test_idx].numpy()
u_obs = u_surface_sensor[test_idx]

initial_guess = [np.random.rand() + 1, np.random.rand()*0.1 + 0.15]

print(f"Index: {test_idx} - True Mass={mu_true_phys[0]:.4f}, True Delta={mu_true_phys[1]:.4f}")
print(f"Initial guess: {initial_guess}")

print("\n--- EXPLORATION ---")
initial_cov_expl = [[0.001, 0], [0, 0.001]]

t0_FOM = perf_counter()

chain_exploration_FOM, cov_learned_FOM = adaptive_metropolis_hastings_fom(
    u_obs=u_obs,
    theta_0=initial_guess,
    N=1000,
    bounds=bounds,
    surface_idx=surface_idx,
    temperature=5.0,
    C_0=initial_cov_expl,
    n_0=200,            # Adjusted n_0
    s_d=2.4,
    n_generations=10,
    h=0.1
)

t_exp_FOM = perf_counter() - t0_FOM
print(f"Exploration Time: {t_exp_FOM:.2f} s")

best_guess_FOM = chain_exploration_FOM[-1]
print(f"Best Guess after exploration: {best_guess_FOM}")

print("\n--- REFINEMENT ---")
cov_for_refinement_FOM = cov_learned_FOM + np.eye(2) * 1e-6

t1_FOM = perf_counter()

chain_refined_FOM, _ = adaptive_metropolis_hastings_fom(
    u_obs=u_obs,
    theta_0=best_guess_FOM,
    N=3000,
    bounds=bounds,
    surface_idx=surface_idx,
    temperature=1.0,
    C_0=cov_for_refinement_FOM,
    n_0=0,
    s_d=0.35,
    n_generations=25,
    h=0.03
)

t_ref_FOM = perf_counter() - t1_FOM
print(f"Refinement Time: {t_ref_FOM:.2f} s")

In [ ]:
print(f">>>FOM\nExploration time:\t{t_exp_FOM}\nRefinement time:\t{t_ref_FOM}")

> FOM (6247)

Exploration time:	339.4052613060003 - Refinement time:	4085.5681142170006

>FOM (6347)

Exploration time:	350.5520346190001 - Refinement time:	4268.049517457001

>FOM (6304)

Exploration time:	355.277866311 - Refinement time:	3933.577092783

>FOM (6265)

Exploration time:	339.96668936900005 - Refinement time:	3899.6581993580003

>FOM (6294)

Exploration time:	325.66602054500004 - Refinement time:	4064.440288521

>FOM (6386)

Exploration time:	370.86808024400034 - Refinement time:	4024.2572852499998

In [ ]:
full_chain_FOM = np.vstack([chain_exploration_FOM, chain_refined_FOM])
split_point_FOM = len(chain_exploration_FOM)
clean_samples_FOM = chain_refined_FOM[500:]

fig1, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(full_chain_FOM[:, 0], color='#1f77b4', linewidth=0.8, alpha=0.9, label='Chain')
axes[0].axvline(x=split_point_FOM, color='black', linestyle='--', linewidth=1.5, label='Refinement')
axes[0].axhline(y=mu_true_phys[0], color='red', linestyle='--', linewidth=2, label='True target')
axes[0].set_title(f"Trace Plot: Mass ({mu_true_phys[0]:.4f})", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Mass")
axes[0].set_xlabel("Iteration")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(full_chain_FOM[:, 1], color='#ff7f0e', linewidth=0.8, alpha=0.9, label='Chain')
axes[1].axvline(x=split_point_FOM, color='black', linestyle='--', linewidth=1.5)
axes[1].axhline(y=mu_true_phys[1], color='red', linestyle='--', linewidth=2, label='True target')
axes[1].set_title(f"Trace Plot: Delta ({mu_true_phys[1]:.4f})", fontsize=14, fontweight='bold')
axes[1].set_ylabel("Delta")
axes[1].set_xlabel("Iteration")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fig2, ax_post = plt.subplots(figsize=(7, 5))

# KDE Plot
sns.kdeplot(
    x=clean_samples_FOM[:, 0],
    y=clean_samples_FOM[:, 1],
    cmap="Blues",
    fill=True,
    thresh=0.05,
    levels=10,
    ax=ax_post
)

ax_post.scatter(mu_true_phys[0], mu_true_phys[1], s=300, c='red', marker='X', label='True target', zorder=10)
ax_post.scatter(clean_samples_FOM[::20, 0], clean_samples_FOM[::20, 1], s=5, c='black', alpha=0.1, label='Samples')
ax_post.set_title("Posterior Distribution", fontsize=14, fontweight='bold')
ax_post.set_xlabel("Mass")
ax_post.set_ylabel("Delta")
ax_post.legend()
ax_post.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
cleaner = 1000
mass_chain_FOM = chain_refined_FOM[cleaner:, 0]
delta_chain_FOM = chain_refined_FOM[cleaner:, 1]

print(f"True mass: {mu_true_phys[0]:.4f}")
print(f"Estimated Mass: {np.mean(mass_chain_FOM):.4f} +/- {np.std(mass_chain_FOM):.4f}")
print(f"Mass relative error: {abs(np.mean(mass_chain_FOM) - mu_true_phys[0])/mu_true_phys[0]:.4%}")

print("-" * 20)
print(f"True Delta: {mu_true_phys[1]:.4f}")
print(f"Estaimted Delta: {np.mean(delta_chain_FOM):.4f} +/- {np.std(delta_chain_FOM):.4f}")
print(f"Delta relative error: {abs(np.mean(delta_chain_FOM) - mu_true_phys[1])/mu_true_phys[1]:.4%}")

test_idx = 6347

True mass: 1.1632 - Estimated Mass: 1.6696 +/- 0.0433 - Mass relative error: 43.5355%

True Delta: 0.2330 - Estaimted Delta: 0.1494 +/- 0.0096 - Delta relative error: 35.9055%

test_idx = 6304

True mass: 1.1811 - Estimated Mass: 1.1837 +/- 0.0299 - Mass relative error: 0.2195%

True Delta: 0.2132 - Estaimted Delta: 0.2152 +/- 0.0098 - Delta relative error: 0.9376%

test_idx = 6265

True mass: 1.6223 - Estimated Mass: 1.5846 +/- 0.1311 - Mass relative error: 2.3228%

True Delta: 0.1962 - Estaimted Delta: 0.1910 +/- 0.0202 - Delta relative error: 2.6628%

test_idx = 6294

True mass: 1.4037 - Estimated Mass: 1.4056 +/- 0.0293 - Mass relative error: 0.1384%

True Delta: 0.2213 - Estaimted Delta: 0.2161 +/- 0.0095 - Delta relative error: 2.3299%

test_idx = 6386

True mass: 1.8839 - Estimated Mass: 1.5765 +/- 0.0382 - Mass relative error: 16.3162%

True Delta: 0.1505 - Estaimted Delta: 0.1847 +/- 0.0087 - Delta relative error: 22.7101%